# 07 — Large Scale Classification Pipeline
**Project:** Syllabus Policy Stance Detection
**Author:** Charitha
**Runtime:** Google Colab (no GPU needed for Stage 1, T4 for Stage 2)

## What this notebook does:
1. Loads full 26,600 sentence dataset from Google Drive
2. Stage 1: Uses Qwen-2.5 72B via Groq to filter policy-relevant sentences
3. Stage 2: Uses fine-tuned RoBERTa to classify stance
4. Compares stance labels vs existing sentiment labels (validation)
5. Weighted analysis using count column
6. Produces paper-ready analysis figures

## Key design decisions:
- Batch processing (25 sentences per API call) to stay within rate limits
- Checkpoint saving after every 500 sentences — safe against Colab disconnects
- Test set sentences excluded from large-scale inference to prevent leakage
- count column used as frequency weight throughout analysis

## Before running:
1. Mount Google Drive
2. Confirm paths in Cell 2
3. No GPU needed until Cell 8 (RoBERTa inference)

In [22]:
# import subprocess
# subprocess.run(['pip', 'install', 'typing_extensions==4.12.2', '--upgrade', '-q'], 
#                capture_output=True)
# print('Fixed! Now restart the kernel.')

Fixed! Now restart the kernel.


In [1]:
# Cell 1 — Environment check (CRC version — no Drive mount needed)
import torch
import os

if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'Mem : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — running on CPU (fine for Stage 1 API calls)')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Working directory: {os.getcwd()}')

No GPU — running on CPU (fine for Stage 1 API calls)
Device: cpu
Working directory: /ihome/mrfrank/chb299/stance_tool_prep/scripts


In [2]:
# Cell 2 — Install dependencies
import subprocess
subprocess.run(['pip', 'install', 'transformers', 'torch', 
                'pandas', 'numpy', 'matplotlib', 'seaborn', '-q', '--user'],
               capture_output=True)
print('Done!')

Done!


In [3]:
# import urllib.request
# try:
#     urllib.request.urlopen('https://api.groq.com', timeout=5)
#     print('Groq accessible!')
# except Exception as e:
#     print(f'Groq blocked: {e}')

Groq blocked: HTTP Error 403: Forbidden


In [3]:
# Cell 3 — Imports
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
warnings.filterwarnings('ignore')

print('Imports done!')

Imports done!


In [4]:
# Cell 4 — Paths and config (CRC version)
import os

# ── CRC Paths ──────────────────────────────────────────────────────────────
RAW_DATA_PATH   = '/ihome/mrfrank/chb299/stance_tool_prep/data/wiki_unique_sentences_clean.csv'
RESULTS_DIR     = '/ihome/mrfrank/chb299/stance_tool_prep/data/results/'
MODELS_DIR      = '/ihome/mrfrank/chb299/stance_tool_prep/data/models/'
ANALYSIS_DIR    = '/ihome/mrfrank/chb299/stance_tool_prep/data/analysis/'
CHECKPOINT_PATH = '/ihome/mrfrank/chb299/stance_tool_prep/data/07_checkpoint.csv'

for d in [RESULTS_DIR, MODELS_DIR, ANALYSIS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
RANDOM_SEED      = 42
LABELS           = ['discouraging', 'conditional', 'encouraging']
LABEL2IDX        = {'discouraging': 0, 'conditional': 1, 'encouraging': 2}
IDX2LABEL        = {0: 'discouraging', 1: 'conditional', 2: 'encouraging'}
BATCH_SIZE       = 16
CHECKPOINT_EVERY = 500
POLICY_THRESHOLD = 0.60

# ── No Groq needed ─────────────────────────────────────────────────────────
# Using local facebook/bart-large-mnli for policy relevance filtering

print('Config set!')
print(f'Data path  : {RAW_DATA_PATH}')
print(f'Results    : {RESULTS_DIR}')
print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Threshold  : {POLICY_THRESHOLD}')

Config set!
Data path  : /ihome/mrfrank/chb299/stance_tool_prep/data/wiki_unique_sentences_clean.csv
Results    : /ihome/mrfrank/chb299/stance_tool_prep/data/results/
Checkpoint : /ihome/mrfrank/chb299/stance_tool_prep/data/07_checkpoint.csv
Threshold  : 0.6


In [5]:
from transformers import pipeline

print('Loading bart-large-mnli...')
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=-1
)
print('Loaded!')

# Test
# Better labels — more descriptive = better zero-shot performance
result = classifier(
    "Wikipedia is not an acceptable source for citations.",
    [
        "this sentence states a rule about whether students can use Wikipedia",
        "this sentence is not about Wikipedia usage rules or policies"
    ]
)
print(result)

Loading bart-large-mnli...


Device set to use cpu


Loaded!
{'sequence': 'Wikipedia is not an acceptable source for citations.', 'labels': ['this sentence states a rule about whether students can use Wikipedia', 'this sentence is not about Wikipedia usage rules or policies'], 'scores': [0.8732630014419556, 0.12673701345920563]}


In [6]:
# Cell 5 — Load dataset with latin1 encoding
import pandas as pd

df = pd.read_csv(
    RAW_DATA_PATH,
    encoding='latin1',
    on_bad_lines='skip',
    engine='python'
)

print(f'Loaded: {len(df):,} sentences')
print(f'Columns: {df.columns.tolist()}')
print()
print('Label distribution:')
print(df['label'].value_counts())
print()
print('Sample rows:')
for _, row in df.head(3).iterrows():
    print(f'  [{row["label"]}] {str(row["sentence"])[:80]}')

Loaded: 26,600 sentences
Columns: ['sentence', 'count', 'sentiment', 'label', 'score']

Label distribution:
label
NEGATIVE    21474
POSITIVE     5126
Name: count, dtype: int64

Sample rows:
  [NEGATIVE] Unlike traditional encyclopedias, Wikipedia entry authors are not required to ha
  [NEGATIVE] Be aware that Wikipedia, Investopedia, and other on-line dictionaries and encycl
  [POSITIVE] i actually had a fun time studying hinduism on wikipedia, which is my topic for 


In [7]:
# ============================================================
# CELL 6 — Load bart-large-mnli for policy relevance filtering
# ============================================================
from transformers import pipeline
import time

print('Loading facebook/bart-large-mnli...')
print('(Model already cached from test — will load instantly)')

classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=-1   # CPU
)

# Labels — descriptive phrasing gives much better zero-shot accuracy
POLICY_LABEL     = 'this sentence states a rule about whether students can use Wikipedia'
NON_POLICY_LABEL = 'this sentence is not about Wikipedia usage rules or policies'
CANDIDATE_LABELS = [POLICY_LABEL, NON_POLICY_LABEL]
POLICY_THRESHOLD = 0.65   # confidence threshold

print('Model loaded!')
print(f'Policy label    : {POLICY_LABEL}')
print(f'Threshold       : {POLICY_THRESHOLD}')
print()

# Validation on known examples
test_cases = [
    ("Wikipedia is not an acceptable source for any assignment.", True),
    ("The assignment is due on Friday at midnight.", False),
    ("Students may consult Wikipedia as a starting point but must not cite it.", True),
    ("Office hours are held on Tuesdays from 2-4pm.", False),
    ("Wikipedia can be used for background research only.", True),
    ("Please bring your textbook to every class.", False),
]

print('Validation on known examples:')
correct = 0
for sentence, expected_policy in test_cases:
    result     = classifier(sentence, CANDIDATE_LABELS)
    is_policy  = result['labels'][0] == POLICY_LABEL and result['scores'][0] >= POLICY_THRESHOLD
    score      = result['scores'][0] if result['labels'][0] == POLICY_LABEL else 1 - result['scores'][0]
    status     = 'CORRECT' if is_policy == expected_policy else 'WRONG  '
    if is_policy == expected_policy:
        correct += 1
    print(f'  [{status}] [{"POLICY" if is_policy else "NOT   "} | {score:.3f}] {sentence[:70]}')

print(f'\nValidation accuracy: {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.0f}%)')
print('Ready to run on full dataset!')

Loading facebook/bart-large-mnli...
(Model already cached from test — will load instantly)


Device set to use cpu


Model loaded!
Policy label    : this sentence states a rule about whether students can use Wikipedia
Threshold       : 0.65

Validation on known examples:
  [CORRECT] [POLICY | 0.976] Wikipedia is not an acceptable source for any assignment.
  [CORRECT] [NOT    | 0.338] The assignment is due on Friday at midnight.
  [CORRECT] [POLICY | 0.971] Students may consult Wikipedia as a starting point but must not cite i
  [CORRECT] [NOT    | 0.356] Office hours are held on Tuesdays from 2-4pm.
  [WRONG  ] [NOT    | 0.588] Wikipedia can be used for background research only.
  [CORRECT] [NOT    | 0.390] Please bring your textbook to every class.

Validation accuracy: 5/6 (83%)
Ready to run on full dataset!


In [8]:
# ============================================================
# CELL 7 — Run Stage 1 on all 26,600 sentences
# ============================================================
import pandas as pd
import os
import time

# Resume from checkpoint if exists
if os.path.exists(CHECKPOINT_PATH):
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    processed_idx = set(df_checkpoint.index[df_checkpoint['policy_relevant'].notna()])
    print(f'Resuming from checkpoint: {len(processed_idx):,} already processed')
else:
    df_checkpoint = df.copy()
    df_checkpoint['policy_relevant']       = None
    df_checkpoint['policy_relevance_score'] = None
    processed_idx = set()
    print('Starting fresh classification...')

todo_idx = [i for i in df.index if i not in processed_idx]

print(f'Total sentences    : {len(df):,}')
print(f'Already processed  : {len(processed_idx):,}')
print(f'Remaining          : {len(todo_idx):,}')
print(f'Est. time on CPU   : ~{len(todo_idx) * 0.35 / 3600:.1f} hours')
print()
print('Starting... (checkpoint saves every 500 sentences)')
print('You can safely close the browser — CRC will keep running')
print()

policy_count = 0
start        = time.time()

for i, idx in enumerate(todo_idx):
    sentence = str(df.loc[idx, 'sentence'])[:512]  # truncate at 512 chars

    try:
        result    = classifier(sentence, CANDIDATE_LABELS)
        top_label = result['labels'][0]
        top_score = result['scores'][0]

        is_policy = top_label == POLICY_LABEL and top_score >= POLICY_THRESHOLD

        # Store policy relevance score (always from policy label perspective)
        policy_score = top_score if top_label == POLICY_LABEL else 1 - top_score

        df_checkpoint.loc[idx, 'policy_relevant']        = 'yes' if is_policy else 'no'
        df_checkpoint.loc[idx, 'policy_relevance_score'] = round(policy_score, 4)

        if is_policy:
            policy_count += 1

    except Exception as e:
        df_checkpoint.loc[idx, 'policy_relevant']        = 'no'
        df_checkpoint.loc[idx, 'policy_relevance_score'] = 0.0

    # Progress every 200 sentences
    if (i + 1) % 200 == 0:
        elapsed = time.time() - start
        rate    = (i + 1) / elapsed
        eta_h   = (len(todo_idx) - i - 1) / rate / 3600
        pct     = policy_count / (i + 1) * 100
        print(f'  {i+1:>6,}/{len(todo_idx):,} | '
              f'Policy: {policy_count:,} ({pct:.1f}%) | '
              f'ETA: {eta_h:.1f}h | '
              f'Speed: {rate:.1f} sent/sec')

    # Save checkpoint every 500
    if (i + 1) % 500 == 0:
        df_checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# Final save
df_checkpoint.to_csv(CHECKPOINT_PATH, index=False)

elapsed_total = time.time() - start
print()
print('='*55)
print('  STAGE 1 COMPLETE')
print('='*55)
print(f'Total processed        : {len(df):,}')
print(f'Policy-relevant found  : {policy_count:,}')
print(f'Policy rate            : {policy_count/len(df)*100:.1f}%')
print(f'Total time             : {elapsed_total/3600:.1f} hours')
print(f'Checkpoint saved       : {CHECKPOINT_PATH}')

Resuming from checkpoint: 18,500 already processed
Total sentences    : 26,600
Already processed  : 18,500
Remaining          : 8,100
Est. time on CPU   : ~0.8 hours

Starting... (checkpoint saves every 500 sentences)
You can safely close the browser — CRC will keep running

   7,600/8,100 | Policy: 2,458 (32.3%) | ETA: 0.2h | Speed: 0.7 sent/sec
   7,800/8,100 | Policy: 2,511 (32.2%) | ETA: 0.1h | Speed: 0.7 sent/sec
   8,000/8,100 | Policy: 2,565 (32.1%) | ETA: 0.0h | Speed: 0.7 sent/sec

  STAGE 1 COMPLETE
Total processed        : 26,600
Policy-relevant found  : 2,593
Policy rate            : 9.7%
Total time             : 3.2 hours
Checkpoint saved       : /ihome/mrfrank/chb299/stance_tool_prep/data/07_checkpoint.csv


In [9]:
# Cell 8 — Filter to policy-relevant sentences
#
# Extract only the policy-relevant sentences for Stage 2 classification.
# These are the sentences Qwen identified as containing Wikipedia policies.

df_policy = df_checkpoint[df_checkpoint['policy_relevant'] == 'yes'].copy()
df_policy = df_policy.reset_index(drop=True)

print(f'Policy-relevant sentences: {len(df_policy):,}')
print(f'Total sentences          : {len(df_checkpoint):,}')
print(f'Policy rate              : {len(df_policy)/len(df_checkpoint)*100:.1f}%')
print()
print('Existing label distribution in policy sentences:')
print(df_policy['label'].value_counts())
print()
print('Sentiment distribution in policy sentences:')
print(df_policy['sentiment'].value_counts())
print()
print('Count stats (frequency across syllabi):')
print(df_policy['count'].describe().round(1))
print()
print('Sample policy sentences:')
for _, row in df_policy.head(5).iterrows():
    print(f'  [{row["label"]} | count={row["count"]}]')
    print(f'  {str(row["sentence"])[:120]}')
    print()

# Save policy-relevant dataset
policy_path = f'{RESULTS_DIR}07_policy_relevant_sentences.csv'
df_policy.to_csv(policy_path, index=False)
print(f'Saved: {policy_path}')

Policy-relevant sentences: 9,686
Total sentences          : 26,600
Policy rate              : 36.4%

Existing label distribution in policy sentences:
label
NEGATIVE    8637
POSITIVE    1049
Name: count, dtype: int64

Sentiment distribution in policy sentences:
sentiment
positive    4356
neutral     2770
negative    2560
Name: count, dtype: int64

Count stats (frequency across syllabi):
count    9686.0
mean        3.1
std        15.5
min         1.0
25%         1.0
50%         1.0
75%         2.0
max       554.0
Name: count, dtype: float64

Sample policy sentences:
  [NEGATIVE | count=554]
  Be aware that Wikipedia, Investopedia, and other on-line dictionaries and encyclopedias are not verifiable sources of re

  [NEGATIVE | count=472]
  Websites, such as Wikipedia or Yahoo, DO NOT necessarily contain reliable facts, documentation, or interpretations, and 

  [POSITIVE | count=422]
  Scholarly Literature Sources and the Use of Wikipedia: University writing and research, and particularly

In [10]:
import pandas as pd

df_check = pd.read_csv('/ihome/mrfrank/chb299/stance_tool_prep/data/07_checkpoint.csv')

print(f'Total rows: {len(df_check):,}')
print()
print('policy_relevant value counts:')
print(df_check['policy_relevant'].value_counts(dropna=False))
print()

# Apply threshold consistently across ALL rows
yes_rows = df_check[df_check['policy_relevant'] == 'yes']
print(f'Rows marked yes: {len(yes_rows):,}')
print()

# Filter by score threshold to get clean results
if 'policy_relevance_score' in df_check.columns:
    clean = df_check[
        (df_check['policy_relevant'] == 'yes') & 
        (df_check['policy_relevance_score'] >= 0.75)
    ]
    print(f'After score >= 0.75 filter: {len(clean):,}')
    
    clean2 = df_check[
        (df_check['policy_relevant'] == 'yes') & 
        (df_check['policy_relevance_score'] >= 0.70)
    ]
    print(f'After score >= 0.70 filter: {len(clean2):,}')
else:
    print('No score column — first session did not save scores')
    print('Need to check which rows have scores and which dont')

Total rows: 26,600

policy_relevant value counts:
policy_relevant
no     16914
yes     9686
Name: count, dtype: int64

Rows marked yes: 9,686

After score >= 0.75 filter: 8,592
After score >= 0.70 filter: 9,103
